In [12]:
from joblib import Parallel,delayed
import sys
import pathlib
import os
from skmap.catalog import DataCatalog
from skmap.loader import TiledDataLoader
from skmap.overlay import SpaceOverlay, SpaceTimeOverlay
from skmap.misc import find_files, GoogleSheet, ttprint
from osgeo.gdal import BuildVRT, SetConfigOption
import random
import pandas as pd
import time
import skmap_bindings as sb
import numpy as np
from shapely.geometry import Point
from geopandas import gpd 
from shapely.geometry import Point
import duckdb

In [ ]:
folder_path = './'

base_path = [f'http://192.168.49.{gaia_id}:8333' for gaia_id in range(30,47)]
GDAL_OPTS = {'GDAL_HTTP_VERSION': '1.0', 'CPL_VSIL_CURL_ALLOWED_EXTENSIONS': '.tif'}
max_ram_mb = 1000000
n_threads = 1

# read in gsheet
gsheet_key = '../stac/gaia-319808-913d36b5fca4.json'
gsheet_url = 'https://docs.google.com/spreadsheets/d/1lNTpzdHBG5dirYj46iBDRJMk_YAV0Um2ovBc8v3dR9w/edit?gid=78425683#gid=78425683'
gsheet = GoogleSheet(gsheet_key, gsheet_url, verbose=False)

years = [2019,2020,2021,2022]
years = [int(ii) for ii in years]

# create catalog
catalog = DataCatalog.create_catalog(catalog_def=gsheet.canopy_height_comparison, years=years, base_path=base_path)

gedi_stac_items=gpd.read_file('https://s3.eu-central-1.wasabisys.com/stac/openlandmap/GEDI02/stac_items_geojson')

def worker(tile):
#for tile in gedi_stac_items.iterrows():
    if os.path.exists(f'{folder_path}/material/ovelayed_{tile[1].item_id}.pq'):
        print(f'{tile[1].item_id} has been done')
        return
 
    start = time.time()
    # create catalog
    catalog = DataCatalog.create_catalog(catalog_def=gsheet.canopy_height_comparison, years=years, base_path=base_path)
    df_duckdb = duckdb.sql(f"""
                            INSTALL httpfs;
                            LOAD httpfs;
                            INSTALL spatial;
                            LOAD spatial;

                            SELECT longitude, latitude, rh95,elev_lowestmode,rh100,rh99,rh98,rh97,rh75,rh50,rh25,sensitivity,night_flag
                            FROM "{tile[1].asset_file}"
                            """)
    df = df_duckdb.df()
    if len(df)>10:
        df=df.sample(int(len(df)*0.1),random_state=1)
        gdf = gpd.GeoDataFrame(
            df, geometry=gpd.points_from_xy(df['longitude'], df['latitude']),crs='EPSG:4326')
    df['year']=tile[1].start_date.year
    df=df.rename(columns={'longitude':'lon','latitude':'lat'})
    try:
        space_time_overlay = SpaceTimeOverlay(
                col_date='year',
                points=df, 
                catalog=catalog,
                verbose=True,
                n_threads=n_threads)

        ovelayed_props_data = space_time_overlay.run(gdal_opts=GDAL_OPTS, max_ram_mb=max_ram_mb, out_file_name=f'{folder_path}/material/ovelayed_{tile[1].item_id}.pq')
        print(f"Overlay for a tile: {(time.time() - start):.2f} s")
    except:
        pass


In [13]:
# overlay the 100th tile of gedi stac
tile =  [i for i in gedi_stac_items.iterrows()][100]

In [14]:
# overlay
worker(tile)

lon=-105_lat=20_year=2019_gedi_l2ab has been done


In [15]:
# parallelize all the tiles in gedi_stac
Parallel(n_jobs=-1)(delayed(worker)(i) for i in gedi_stac_items.iterrows())

No points to overlay for year 2019, removing it from the catalog
No points to overlay for year 2020, removing it from the catalog
No points to overlay for year 2021, removing it from the catalog
No points to overlay for year 2022, removing it from the catalog
[07:59:32] Running the overlay for common
lon=-10_lat=5_year=2021_gedi_l2ab has been done
lon=-10_lat=45_year=2021_gedi_l2ab has been done
lon=-10_lat=5_year=2022_gedi_l2ab has been done
lon=-10_lat=30_year=2020_gedi_l2ab has been done
lon=-10_lat=20_year=2021_gedi_l2ab has been done
lon=-100_lat=35_year=2019_gedi_l2ab has been done
lon=-105_lat=30_year=2019_gedi_l2ab has been done
lon=-10_lat=30_year=2022_gedi_l2ab has been done
lon=-10_lat=45_year=2022_gedi_l2ab has been done
lon=-10_lat=5_year=2019_gedi_l2ab has been done
lon=-10_lat=35_year=2020_gedi_l2ab has been done
lon=-10_lat=15_year=2019_gedi_l2ab has been done
lon=-100_lat=30_year=2021_gedi_l2ab has been done
lon=-105_lat=25_year=2021_gedi_l2ab has been done
lon=-10_lat

Exception ignored in: <module 'collections.abc' from '/opt/conda/lib/python3.8/collections/abc.py'>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
KeyboardInterrupt: 
Exception ignored in: <module 'collections.abc' from '/opt/conda/lib/python3.8/collections/abc.py'>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
KeyboardInterrupt: 
Process LokyProcess-483:
Traceback (most recent call last):
  File "/opt/conda/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/opt/conda/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/opengeohub/.local/lib/python3.8/site-packages/joblib/externals/loky/process_executor.py", line 478, in _process_worker
    _process_reference_size = _get_memory_usage(pid, force_gc=True)
  File "/home/opengeohub/.local/lib/python3.8/site-packages/joblib/externals/loky/process_executor.py", line 109, in _get_mem

core/api.py", line 47, in <module>
    from pandas.core.groupby import (
  File "/opt/conda/lib/python3.8/site-packages/pandas/core/groupby/__init__.py", line 1, in <module>
    from pandas.core.groupby.generic import (
  File "/opt/conda/lib/python3.8/site-packages/pandas/core/groupby/generic.py", line 77, in <module>
    from pandas.core.frame import DataFrame
  File "/opt/conda/lib/python3.8/site-packages/pandas/core/frame.py", line 182, in <module>
    from pandas.core.generic import NDFrame
  File "/opt/conda/lib/python3.8/site-packages/pandas/core/generic.py", line 138, in <module>
    from pandas.core import (
KeyboardInterrupt

Traceback (most recent call last):
  File "/home/opengeohub/.local/lib/python3.8/site-packages/joblib/externals/loky/process_executor.py", line 426, in _process_worker
    call_item = call_queue.get(block=True, timeout=timeout)
  File "/opt/conda/lib/python3.8/multiprocessing/queues.py", line 116, in get
    return _ForkingPickler.loads(res)
  File "/hom

<frozen importlib._bootstrap>:219: RuntimeWarning: Cython module failed to patch module with custom type

KeyboardInterrupt



Traceback (most recent call last):
  File "/home/opengeohub/.local/lib/python3.8/site-packages/joblib/externals/loky/process_executor.py", line 426, in _process_worker
    call_item = call_queue.get(block=True, timeout=timeout)
  File "/opt/conda/lib/python3.8/multiprocessing/queues.py", line 116, in get
    return _ForkingPickler.loads(res)
  File "/home/opengeohub/src/scikit-map/skmap/__init__.py", line 4, in <module>
    from skmap.misc import ttprint
  File "/home/opengeohub/src/scikit-map/skmap/misc.py", line 13, in <module>
    import geopandas as gp
  File "/opt/conda/lib/python3.8/site-packages/geopandas/__init__.py", line 1, in <module>
    from geopandas._config import options  # noqa
  File "/opt/conda/lib/python3.8/site-packages/geopandas/_config.py", line 109, in <module>
    default_value=_default_use_pygeos(),
  File "/opt/conda/lib/python3.8/site-packages/geopandas/_config.py", line 95, in _default_use_pygeos
    import geopandas._compat as compat
  File "/opt/conda/lib

<frozen importlib._bootstrap>:219: RuntimeWarning: Cython module failed to patch module with custom type
Process LokyProcess-449:
KeyboardInterrupt

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/opengeohub/.local/lib/python3.8/site-packages/joblib/externals/loky/process_executor.py", line 463, in _process_worker
    r = call_item()
  File "/home/opengeohub/.local/lib/python3.8/site-packages/joblib/externals/loky/process_executor.py", line 291, in __call__
    return self.fn(*self.args, **self.kwargs)
  File "/home/opengeohub/.local/lib/python3.8/site-packages/joblib/parallel.py", line 589, in __call__
    return [func(*args, **kwargs)
  File "/home/opengeohub/.local/lib/python3.8/site-packages/joblib/parallel.py", line 589, in <listcomp>
    return [func(*args, **kwargs)
  File "/tmp/ipykernel_93490/2144903300.py", line 19, in worker
RuntimeError: Query interrupted

During handling of the above exception, another 